# Notebook 06: MH-DDPM Fine-tuning and Semi-supervised Model Enhancement
## Fine-tune DDPM on real contaminated spectra and use synthetic positives to enhance Anomaly Detectors

In [ ]:
import numpy as np, pandas as pd, struct
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch; torch.manual_seed(42)
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix
import sys

BASE = Path('/run/media/sham/AI_/ai-stack/projects/biopharma-contamination-detection')
sys.path.append(str(BASE))
from src.anomaly_detection import AutoencoderDetector, OneClassSVMDetector, ModelConfig
from src.validation import ValidationPipeline

OUT = BASE / 'data' / 'processed'
FIG = BASE / 'figures'; FIG.mkdir(exist_ok=True)
MODELS = BASE / 'models'; MODELS.mkdir(exist_ok=True)
df = pd.read_parquet(OUT / 'real_dataset.parquet')

In [ ]:
def unpack(row):
    n = row['n_wl']; data = struct.unpack(f'{n*2}d', row['spectrum_bytes'])
    return np.array(data[::2]), np.array(data[1::2])

target_wl = np.arange(230, 610, 1)
def interp(row): return np.interp(target_wl, *unpack(row))

X_all = np.vstack(df.apply(interp, axis=1).values)
y_all = df['label'].values

X_clean = X_all[y_all == 0]
X_contam = X_all[y_all == 1]

print(f'Clean spectra: {X_clean.shape}')
print(f'Contaminated spectra: {X_contam.shape}')

## 1. Train/Fine-tune DDPM on Real Contaminated Data

In [ ]:
class SimpleDDPM(nn.Module):
    def __init__(self, dim, timesteps=100):
        super().__init__()
        self.T = timesteps
        self.beta = torch.linspace(1e-4, 0.02, timesteps)
        self.alpha = 1 - self.beta
        self.alpha_bar = torch.cumprod(self.alpha, 0)
        self.net = nn.Sequential(
            nn.Linear(dim + 1, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
            nn.Linear(256, dim)
        )
    def forward(self, x, t):
        t_norm = t.float() / self.T
        return self.net(torch.cat([x, t_norm.unsqueeze(1)], dim=1))
    
    @torch.no_grad()
    def sample(self, n_samples, dim):
        x = torch.randn(n_samples, dim)
        for t in reversed(range(self.T)):
            z = torch.randn_like(x) if t > 0 else 0
            t_tensor = torch.full((n_samples,), t, dtype=torch.long, device=x.device)
            eps = self(x, t_tensor)
            x = (x - (1-self.alpha[t])/torch.sqrt(1-self.alpha_bar[t]) * eps) / torch.sqrt(self.alpha[t]) + torch.sqrt(self.beta[t]) * z
        return x

device = 'cuda' if torch.cuda.is_available() else 'cpu'
ddpm = SimpleDDPM(X_contam.shape[1], timesteps=100).to(device)
opt = torch.optim.Adam(ddpm.parameters(), lr=1e-3)

X_t = torch.FloatTensor(X_contam).to(device)
dl = DataLoader(TensorDataset(X_t), batch_size=32, shuffle=True)

print("Training DDPM...")
for epoch in range(50):
    total_loss = 0
    for (batch,) in dl:
        opt.zero_grad()
        t = torch.randint(0, ddpm.T, (batch.shape[0],), device=device)
        noise = torch.randn_like(batch)
        x_noisy = torch.sqrt(ddpm.alpha_bar[t]) * batch + torch.sqrt(1 - ddpm.alpha_bar[t]) * noise
        pred = ddpm(x_noisy, t)
        loss = nn.MSELoss()(pred, noise)
        loss.backward(); opt.step()
        total_loss += loss.item()
    if epoch % 10 == 0: print(f'  Epoch {epoch}: loss={total_loss/len(dl):.6f}')

torch.save(ddpm.state_dict(), MODELS / 'ddpm_real_finetuned.pt')

## 2. Generate Synthetic Positives

In [ ]:
n_synth = 500
synthetic = ddpm.sample(n_synth, X_contam.shape[1]).cpu().numpy()
synthetic = np.maximum(0, synthetic) # Ensure physical validity
print(f'Generated {len(synthetic)} synthetic contaminated spectra')

plt.figure(figsize=(10, 5))
plt.plot(target_wl, X_contam[0], label='Real Contaminated', color='red', alpha=0.5)
plt.plot(target_wl, synthetic[0], label='Synthetic Contaminated', color='blue', alpha=0.5)
plt.legend(); plt.title("Spectral Comparison"); plt.show()

## 3. Baseline Anomaly Detection (Clean-only)

In [ ]:
config = ModelConfig(ae_epochs=50)
ae_baseline = AutoencoderDetector(X_clean.shape[1], config)
print("Training baseline Autoencoder...")
ae_baseline.fit(X_clean, verbose=False)

ocsvm_baseline = OneClassSVMDetector(config)
print("Training baseline OCSVM...")
ocsvm_baseline.fit(X_clean)

def eval_model(model, X, y, name):
    scores = model.predict_proba(X)
    auc = roc_auc_score(y, scores)
    print(f"{name} ROC-AUC: {auc:.4f}")
    return auc

auc_ae_pre = eval_model(ae_baseline, X_all, y_all, "AE Baseline")
auc_ocsvm_pre = eval_model(ocsvm_baseline, X_all, y_all, "OCSVM Baseline")

## 4. Semi-supervised Fine-tuning with Synthetic Positives
We use the synthetic positives to refine the decision boundary.

In [ ]:
print("Refining Autoencoder with synthetic positives (Classification Head)...")
# Create a simple MLP classifier using AE features
z_clean = ae_baseline.get_latent_representation(X_clean)
z_synth = ae_baseline.get_latent_representation(synthetic)

X_semi = np.vstack([z_clean, z_synth])
y_semi = np.concatenate([np.zeros(len(z_clean)), np.ones(len(z_synth))])

from sklearn.linear_model import LogisticRegression
clf = LogisticRegression().fit(X_semi, y_semi)

def refined_predict_proba(X):
    z = ae_baseline.get_latent_representation(X)
    return clf.predict_proba(z)[:, 1]

scores_refined = refined_predict_proba(X_all)
auc_ae_post = roc_auc_score(y_all, scores_refined)
print(f"Refined AE ROC-AUC: {auc_ae_post:.4f}")

In [ ]:
print("Refining OCSVM using synthetic positives for parameter selection...")
best_auc = 0
best_nu = 0.05

for nu in [0.01, 0.05, 0.1, 0.2]:
    tmp_config = ModelConfig(ocsvm_nu=nu)
    tmp_model = OneClassSVMDetector(tmp_config)
    tmp_model.fit(X_clean)
    
    # Validate on synthetic positives + portion of clean
    y_val = np.concatenate([np.zeros(100), np.ones(len(synthetic))])
    X_val = np.vstack([X_clean[:100], synthetic])
    
    auc = roc_auc_score(y_val, tmp_model.predict_proba(X_val))
    if auc > best_auc:
        best_auc = auc
        best_nu = nu

print(f"Best nu found using synthetic data: {best_nu}")
ocsvm_refined = OneClassSVMDetector(ModelConfig(ocsvm_nu=best_nu))
ocsvm_refined.fit(X_clean)
auc_ocsvm_post = eval_model(ocsvm_refined, X_all, y_all, "Refined OCSVM")

## 5. Performance Reporting and Artifact Saving

In [ ]:
results = pd.DataFrame({
    'Model': ['Autoencoder', 'One-Class SVM'],
    'Baseline AUC': [auc_ae_pre, auc_ocsvm_pre],
    'Refined AUC': [auc_ae_post, auc_ocsvm_post],
    'Improvement': [auc_ae_post - auc_ae_pre, auc_ocsvm_post - auc_ocsvm_pre]
})

print("\nFinal Comparison:")
print(results)
results.to_csv(BASE / 'output' / 'finetune_results.csv', index=False)

ae_baseline.save(str(MODELS / 'ae_baseline.pt'))
ocsvm_baseline.save(str(MODELS / 'ocsvm_baseline.pkl'))
ocsvm_refined.save(str(MODELS / 'ocsvm_refined.pkl'))
print(f"\nArtifacts saved to {MODELS}")